In [54]:
import ollama
import pandas as pd
from tqdm import tqdm
#from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import re

In [55]:
MODEL_NAME = 'gemma3:4b'
PROGRAM_NAME = "ALL"

### Load Data from USA Spending, prep it for next steps

In [56]:
#d = pd.read_csv('../Assistance_PrimeAwardSummaries_2025-06-04_H17M57S53_1.csv')
d = pd.read_excel('../OMDGPublicDatasetFY2024.xlsx')
d.head()

,Legal Name,City,State,Zip 5,Congressional District,Amount Requested,Project Type,Project Title,Project Summary
0,Venture Milk LLC,Slocomb,AL,36375,2,950000.0,Processing Capacity Expansion,Expanding Deep South Organic Dairy Processing ...,This project increases output and lowers the c...
1,Agoge Life Inc.,Phoenix,AZ,85044,4,2054000.0,Market Development and Promotion,Organic Hemp Protein Isolate - Market Developm...,"Agoge, a sustainable organic product-focused c..."
2,Craig Schmitt,PEORIA,AZ,85383,8,98283.0,Simplified Equipment,Expand Organic Grain Processing for NE Montana,The Schmitt’s Farm converted to 100% organic i...
3,Rumiano Cheese Company,Willows,CA,95988,1,3000000.0,Processing Capacity Expansion,Expanding Market Opportunities for Organic Dai...,Dairy is the largest organic commodity in the ...
4,Heal the Earth,Cardiff by the Sea,CA,92007,9,2473795.0,Processing Capacity Expansion,Improving the Viability of the Domestic Organi...,"Heal the Earth at Wild Acres Farm, a San Diego..."


In [57]:
list(d.columns)

['Legal Name',
 'City',
 'State',
 'Zip 5',
 'Congressional District',
 'Amount Requested',
 'Project Type',
 'Project Title',
 'Project Summary']

In [58]:
# trim to just FAIN, amount, and description
#d = d[['award_id_fain', 'total_funding_amount', 'cfda_numbers_and_titles', 'prime_award_base_transaction_description']]
d = d[['Legal Name', 'Amount Requested', 'Project Summary']]
d.head()

,Legal Name,Amount Requested,Project Summary
0,Venture Milk LLC,950000.0,This project increases output and lowers the c...
1,Agoge Life Inc.,2054000.0,"Agoge, a sustainable organic product-focused c..."
2,Craig Schmitt,98283.0,The Schmitt’s Farm converted to 100% organic i...
3,Rumiano Cheese Company,3000000.0,Dairy is the largest organic commodity in the ...
4,Heal the Earth,2473795.0,"Heal the Earth at Wild Acres Farm, a San Diego..."


### Get program context info

In [59]:
omdg_context = """
The Organic Market Development Grant (OMDG) program supports the development of new and expanded organic markets to help
increase the consumption of domestic organic agricultural commodities. The program focuses on building and expanding capacity
for certified organic production, aggregation, processing, manufacturing, storing, transporting, wholesaling, distribution, 
and development of consumer markets. OMDG aims to increase the availability and demand for domestically produced organic 
agricultural products and address the critical need for additional market paths.

AMS will give priority consideration to projects addressing specific pinpointed market needs for organic grains and livestock 
feed, organic dairy, organic fibers, organic legumes and other rotational crops, and organic ingredients currently unavailable 
in organic form.
"""

In [60]:
# Prompt template
def format_prompt(description: str, program: str, context: str) -> str:
    return f"""
    You are a helpful assistant. Re-write the following PROJECT DESCRIPTION to make it a single paragraph. 
    While the PROJECT DESCRIPTION is the main source of information you should use for this task, you can also
    use the information in the PROJECT_CONTEXT section below to inform your response. Lastly, follow the INSTRUCTIONS 
    listed below very carefully. Deviation from these instructions will be strongly penalized.

    PROJECT DESCRIPTION:
    \"\"\"
    {description}
    \"\"\"

    PROJECT CONTEXT:
    This project is part of the {program} program. Here is additional information about the project: {context}.
    
    INSTRUCTIONS:
    - Return a single paragraph with complete sentences, totaling 370 words or fewer.
    - Start the first sentence of each summary with the name of the organization listed in the PROJECT DESCRIPTION.
    - Make the summary sound professional, clear, and with correct grammar, spelling, and punctuation
    - Include the following information in the summary, in this order:
        1. Purpose of the project
        2. Items being purchased or built (if applicable)
        3. Anticipated outcomes
        4. Beneficiaries (the people or communities helped by the project)
    - Describe the beneficiary populations in general terms using words such as "farmers", "consumers", "ranchers", 
        "processors", and/or "stakeholders"
    - Exclude the following words: "diversity", "underserved", "under-served", "minority", "urban", "inner city",
        "woman owned", "climate smart", "conservation", "climate change", and "biodiversity", and anything else that
        might be considered (rightly or not) to be DEI-related.
    - Exclude any discussion of training.
    - Do not include anything about the "PROJECT CONTEXT" (i.e., don't mention the program that this project is part of); focus
        only on the project in the PROJECT DESCRIPTION.
    Use complete sentences.
    - Do not include anything about your instructions (i.e. omit anything like "as a helpful assistant..." or "Here is the 
        summary I re-wrote based on my instructions")
    - Remove subjective terms like "delicious" and keep the description as factual and objective as possible.
    - If the PROJECT DESCRIPTION is shorter than 40 words, just return "Insufficient information"
    - If the PROJECT DESCRIPTION is a single sentence, return "Insufficient information"

    """

In [61]:
# Summarize a batch of projects with context
def summarize_batch(descriptions: list[str], model: str =  MODEL_NAME) -> list[str]:
    summaries = []
    for desc in tqdm(descriptions):
        prompt = format_prompt(desc, 'OMDG', omdg_context)
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        summaries.append(response["message"]["content"].strip())
    return summaries

In [62]:
#d = d.sample(n=1)
summaries = summarize_batch(d["Project Summary"].tolist(), model=MODEL_NAME)

100%|█████████████████████████████████████████| 107/107 [17:32<00:00,  9.84s/it]


In [63]:
d['llm_summary'] = summaries

In [64]:
print(d.iloc[0]['Project Summary'])

This project increases output and lowers the costs of a small organic dairy processing plant serving underserved producers in economically distressed areas. By supporting a full-time sales position, it expands market access both for existing organic dairy producers and for dairy producers transitioning to organic. It does this through co-packing, two processor-affiliated brands, and other marketing channels it has developed and will continue to expand. Plant upgrades include adding packaging options optimized for food service and food product manufacturers that represent over a third of the national demand for dairy products. The desire central outcome of these activities is to develop product market matches that successfully generate sustained income for organic dairy producers by meeting the needs of existing sales outlets more efficiently and completely.


In [66]:
print(d.iloc[0]['llm_summary'])

This project aims to increase output and lower costs for a small organic dairy processing plant, supporting farmers and processors within economically distressed areas. The core activities involve expanding market access for both established organic dairy producers and those transitioning to organic practices through co-packing initiatives and two processor-affiliated brands, alongside the expansion of marketing channels. A key component of this project is the addition of plant upgrades, specifically incorporating packaging options optimized for food service and food product manufacturers, representing a significant portion – over a third – of national demand for dairy products. The anticipated outcome is the development of successful product market matches that will sustainably generate income for organic dairy producers by more efficiently and completely meeting the requirements of existing sales outlets. These activities will directly benefit farmers, ranchers, and processors, contr

In [67]:
d.to_csv(f'OMDG_llm_description_rewrite_testing_{re.sub(r"[^a-zA-Z0-9]", "", MODEL_NAME)}.csv')